# NB09 — 제약 레이어 통합 점수 (Constraint Layer Scoring)

H3 셀 단위로 **5개 물리적 제약 레이어**를 통합하여 버티포트·드론 배송 허브 후보지 분석용 종합 점수를 산출합니다.

## 설계 원칙

`constraint_score`는 **물리적 환경 품질**만 반영합니다.  
공역제한(관제권)은 규제 레이어로 `hard_exclusion_flag`를 통해 별도 추적합니다.  
이를 통해 "물리적으로는 최적이나 규제 제약이 있는 셀"과 "물리적으로 열악한 셀"을 구분합니다.

## 레이어 구성

| # | 레이어 | 소스 | 가중치 |
|---|--------|------|--------|
| 1 | 공역제한 (Airspace) | **VWorld WFS 관제권 `lt_c_aisctrc`** (NB07d) | *별도 — `hard_exclusion_flag`* |
| 2 | 장애물 (Obstacle) | seongnam_buildings.gpkg | **0.25** |
| 3 | 소음 프록시 (Noise proxy) | 건물 밀도 기반 | **0.1875** |
| 4 | 지형 (Terrain slope) | seongnam_slope_b010.tif | **0.25** |
| 5 | 로봇 접근성 (Robot access) | robot_access_grid.gpkg | **0.1875** |
| 6 | 기상 (Weather risk) | weather_risk_grid.gpkg | **0.125** |

```
constraint_score = 0.25   × score_obstacle
                 + 0.1875 × score_noise_proxy
                 + 0.25   × score_terrain
                 + 0.1875 × score_robot
                 + 0.125  × score_weather

score_airspace → hard_exclusion_flag (규제 레이어, 별도 추적)
```

## 공역 데이터 우선순위

1. **`processed/airspace_constraint.gpkg`** (layer=`airspace_constraint`) — NB07d가 생성한 VWorld WFS 관제권 공식 파일 ← **현재 소스**
2. VWorld WFS 직접 요청 (`lt_c_aisctrc`) — STEP 1 파일이 없을 때
3. 서울공항 CTR 프록시 — 두 경우 모두 실패 시 (민감도 분석 전용, 스코어링에 미반영)

In [1]:
import os
import json
import warnings
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import h3
from pathlib import Path
from datetime import datetime
from shapely.geometry import Point, box, Polygon, mapping
from shapely.ops import unary_union

warnings.filterwarnings('ignore')

# Robustly resolve project root whether run from 02_analysis or repo root
_cwd = Path.cwd()
if _cwd.name == '02_analysis':
    BASE = _cwd.parent
elif (_cwd / '02_analysis').exists():
    BASE = _cwd
else:
    BASE = _cwd.parent
PROC = BASE / 'processed'
DATA = BASE / '00_data'

print(f'Project root : {BASE}')
print(f'Processed    : {PROC}')
print(f'Run time     : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


Project root : C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset
Processed    : C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset\processed
Run time     : 2026-05-12 09:39:11


In [2]:
demand_gpkg = PROC / 'demand_grid.gpkg'
if not demand_gpkg.exists():
    raise FileNotFoundError(f'demand_grid.gpkg not found: {demand_gpkg}')

hex_gdf = gpd.read_file(demand_gpkg)
print(f'Base H3 grid : {len(hex_gdf)} cells   CRS={hex_gdf.crs}')

# Alias h3_id → h3_index for compatibility
if 'h3_index' not in hex_gdf.columns and 'h3_id' in hex_gdf.columns:
    hex_gdf['h3_index'] = hex_gdf['h3_id']

KEEP_COLS = ['h3_index', 'lat', 'lon', 'CSV_ADMI_CD', 'ADM_NM', 'GU_NM', 'Ds', 'demand_grade']
missing_base = [c for c in KEEP_COLS if c not in hex_gdf.columns]
if missing_base:
    print(f'  WARNING — missing base cols: {missing_base}')
else:
    print(f'  All base columns present.')

# Reproject to EPSG:5179 for spatial ops (Korean TM)
hex_5179 = hex_gdf.to_crs('EPSG:5179')
print(f'  Reprojected to EPSG:5179 for spatial calculations.')


Base H3 grid : 1947 cells   CRS=EPSG:4326
  All base columns present.
  Reprojected to EPSG:5179 for spatial calculations.


In [3]:
# ═══════════════════════════════════════════════════════
#  공역제한 (Airspace Restriction) — score_airspace
# ═══════════════════════════════════════════════════════

VWORLD_KEY  = 'CA689AF7-5C7C-37D4-8973-372AEA113858'
SEONGNAM_BOX = '127.05,37.33,127.20,37.52'

# Seoul Airbase CTR proxy (공식 데이터 없을 때 최후 폴백)
AIRBASE_LON, AIRBASE_LAT = 127.1141, 37.4449
AIRBASE_RADIUS_M = 9300   # 9.3 km CTR approximation

AIRSPACE_PROXY_FLAG  = False
airspace_gdf         = None
airspace_source      = 'none'
airspace_data_status = 'not_checked'

# ── STEP 1: NB07d 공식 관제권 파일 우선 로드 ──────────
# NB07d_airspace_control_zone_vworld.ipynb 가 생성하는
# airspace_constraint.gpkg 를 명시적으로 가장 먼저 확인합니다.
# 이 파일이 있으면 keyword 스캔 없이 즉시 로드합니다.
CONSTRAINT_FILE = PROC / 'airspace_constraint.gpkg'

if CONSTRAINT_FILE.exists():
    try:
        airspace_gdf = gpd.read_file(CONSTRAINT_FILE, layer='airspace_constraint')
        airspace_source = 'local:airspace_constraint.gpkg (VWorld WFS 관제권)'
        airspace_data_status = 'official_vworld_wfs_관제권'
        print(f'[Airspace] 공식 관제권 파일 로드 성공')
        print(f'  파일   : {CONSTRAINT_FILE.name}')
        print(f'  피처 수: {len(airspace_gdf)}')
        print(f'  CRS    : {airspace_gdf.crs}')
        if 'airspace_type' in airspace_gdf.columns:
            print(f'  airspace_type: {airspace_gdf["airspace_type"].unique().tolist()}')
        if 'source_layer' in airspace_gdf.columns:
            print(f'  source_layer : {airspace_gdf["source_layer"].unique().tolist()}')
    except Exception as e:
        print(f'[Airspace] airspace_constraint.gpkg 로드 실패: {e}')
        print('  → VWorld API 시도로 넘어갑니다.')
else:
    print('[Airspace] airspace_constraint.gpkg 없음')
    print('  NB07d_airspace_control_zone_vworld.ipynb 를 먼저 실행하세요.')
    print('  → VWorld API 폴백으로 넘어갑니다.')

# ── STEP 2: VWorld WFS 직접 시도 (STEP 1 실패 시) ─────
if airspace_gdf is None:
    print('\n[Airspace] VWorld WFS 관제권 직접 요청 시도')
    # WFS EPSG:4326 bbox 순서: ymin,xmin,ymax,xmax
    _bbox_parts = SEONGNAM_BOX.split(',')  # lon_min,lat_min,lon_max,lat_max
    _wfs_bbox = f'{_bbox_parts[1]},{_bbox_parts[0]},{_bbox_parts[3]},{_bbox_parts[2]}'
    _wfs_params = {
        'service': 'WFS', 'request': 'GetFeature', 'version': '1.1.0',
        'typename': 'lt_c_aisctrc', 'output': 'application/json',
        'srsname': 'EPSG:4326', 'bbox': _wfs_bbox,
        'maxFeatures': 1000, 'key': VWORLD_KEY,
    }
    try:
        _resp = requests.get('https://api.vworld.kr/req/wfs', params=_wfs_params, timeout=20)
        _ct = _resp.headers.get('Content-Type', '')
        if _resp.status_code == 200 and 'json' in _ct:
            _data = _resp.json()
            _feats = _data.get('features', [])
            if _feats:
                airspace_gdf = gpd.GeoDataFrame.from_features(_feats, crs='EPSG:4326')
                airspace_source = 'vworld_wfs:lt_c_aisctrc'
                airspace_data_status = 'official_vworld_wfs_관제권'
                print(f'  VWorld WFS 성공: {len(airspace_gdf)} features')
            else:
                print(f'  VWorld WFS: 피처 0개 반환')
        else:
            print(f'  VWorld WFS: HTTP {_resp.status_code} / Content-Type={_ct}')
    except Exception as e:
        print(f'  VWorld WFS 요청 실패: {e}')

# ── STEP 3: Seoul Airbase proxy fallback ───────────────
if airspace_gdf is None:
    print('\n[Airspace] 공식 데이터 없음 — 서울공항 CTR 프록시 생성 (민감도 분석 전용)')
    ctr_center = gpd.GeoDataFrame(
        geometry=[Point(AIRBASE_LON, AIRBASE_LAT)], crs='EPSG:4326'
    ).to_crs('EPSG:5179')
    ctr_buffer = ctr_center.buffer(AIRBASE_RADIUS_M)
    airspace_proxy_gdf   = gpd.GeoDataFrame(geometry=ctr_buffer, crs='EPSG:5179').to_crs('EPSG:4326')
    AIRSPACE_PROXY_FLAG  = True
    airspace_source      = 'proxy:seoul_airbase_ctr'
    airspace_data_status = 'official_missing_proxy_created'
    print(f'  프록시 중심: ({AIRBASE_LON}, {AIRBASE_LAT})  반경: {AIRBASE_RADIUS_M/1000:.1f} km')

# ── STEP 4: compute score_airspace ────────────────────
if not AIRSPACE_PROXY_FLAG and airspace_gdf is not None:
    # 공식 데이터: H3 셀과 공역 폴리곤의 교차 여부로 점수 부여
    # airspace_constraint.gpkg 는 EPSG:5179 로 저장되어 있으므로
    # to_crs('EPSG:5179') 는 이미 5179이면 no-op 으로 안전하게 처리됩니다.
    air_5179 = airspace_gdf.to_crs('EPSG:5179')
    restricted_union = unary_union(air_5179.geometry)
    intersects_restricted = hex_5179.geometry.intersects(restricted_union)
    hex_gdf['score_airspace']      = np.where(intersects_restricted, 0.0, 1.0)
    hex_gdf['airspace_method']     = 'official_vworld_wfs_관제권'
    hex_gdf['airspace_proxy_flag'] = False
    n_restricted = int(intersects_restricted.sum())
    print(f'  score_airspace: {n_restricted}개 셀 관제권 내 (score=0.0), '
          f'{len(hex_gdf) - n_restricted}개 셀 외부 (score=1.0)')
else:
    # 프록시: score_airspace = 1.0 (민감도 분석용 proxy 컬럼만 별도 계산)
    hex_gdf['score_airspace']      = 1.0
    hex_gdf['airspace_method']     = 'proxy_sensitivity_only'
    hex_gdf['airspace_proxy_flag'] = True

    proxy_5179 = airspace_proxy_gdf.to_crs('EPSG:5179')
    proxy_union = unary_union(proxy_5179.geometry)
    in_proxy    = hex_5179.geometry.intersects(proxy_union)
    hex_gdf['score_airspace_proxy'] = np.where(in_proxy, 0.0, 1.0)
    n_proxy = int(in_proxy.sum())
    print(f'  score_airspace_proxy: {n_proxy}개 셀이 CTR 프록시 반경 내')

hex_gdf['airspace_data_status'] = airspace_data_status
print(f'\n  공역 소스: {airspace_source}')
print(f'  score_airspace 분포: {hex_gdf["score_airspace"].value_counts().to_dict()}')

[Airspace] 공식 관제권 파일 로드 성공
  파일   : airspace_constraint.gpkg
  피처 수: 2
  CRS    : EPSG:5179
  airspace_type: ['관제권']
  source_layer : ['lt_c_aisctrc']
  score_airspace: 1688개 셀 관제권 내 (score=0.0), 259개 셀 외부 (score=1.0)

  공역 소스: local:airspace_constraint.gpkg (VWorld WFS 관제권)
  score_airspace 분포: {0.0: 1688, 1.0: 259}


In [4]:
# ═══════════════════════════════════════════════════════
#  장애물 위험도 (Obstacle Risk) — score_obstacle
# ═══════════════════════════════════════════════════════

bldg_path = PROC / 'seongnam_buildings.gpkg'
if not bldg_path.exists():
    raise FileNotFoundError(f'seongnam_buildings.gpkg not found: {bldg_path}')

# Load buildings in EPSG:5179 — try known layer name, fallback to default
try:
    bldg = gpd.read_file(bldg_path, layer='buildings_5179')
    layer_name = 'buildings_5179'
except Exception:
    try:
        from pyogrio import list_layers
        layers_bldg = [l[0] for l in list_layers(str(bldg_path))]
        layer_name = layers_bldg[0]
    except Exception:
        layer_name = None
    bldg = gpd.read_file(bldg_path) if layer_name is None else gpd.read_file(bldg_path, layer=layer_name)
print(f'  Layer used: {layer_name}')
if bldg.crs and bldg.crs.to_epsg() != 5179:
    bldg = bldg.to_crs('EPSG:5179')
print(f'  Buildings loaded: {len(bldg):,}  CRS={bldg.crs}')
print(f'  Columns: {list(bldg.columns)}')

# Identify high-rise buildings
if '지상층수' in bldg.columns:
    bldg['_floors'] = pd.to_numeric(bldg['지상층수'], errors='coerce')
    hr_mask  = bldg['_floors'] >= 15
    floor_method = '지상층수 >= 15'
elif '높이_m' in bldg.columns:
    bldg['_floors'] = None
    hr_mask  = pd.to_numeric(bldg['높이_m'], errors='coerce') >= 45
    floor_method = '높이_m >= 45'
elif 'height' in bldg.columns:
    bldg['_floors'] = None
    hr_mask  = pd.to_numeric(bldg['height'], errors='coerce') >= 45
    floor_method = 'height >= 45'
else:
    # Large footprint as proxy (>=1000 m² ≈ large building)
    bldg['_floors'] = None
    hr_mask  = bldg.geometry.area >= 1000
    floor_method = 'footprint area >= 1000 m² (no floor data)'
    print(f'  WARNING: no floor/height column found — using footprint proxy')

highrise = bldg[hr_mask].copy()
all_bldg  = bldg.copy()
print(f'  High-rise criterion: {floor_method}')
print(f'  High-rise buildings: {len(highrise):,} / {len(bldg):,}')

# Buffer H3 cells by 200 m for intersection
OBSTACLE_BUFFER_M = 200
hex_buf = hex_5179.copy()
hex_buf['geometry'] = hex_5179.geometry.buffer(OBSTACLE_BUFFER_M)

# Count high-rise buildings per cell (right join keeps all cells)
if len(highrise) > 0:
    hr_join = gpd.sjoin(
        highrise[['geometry']],
        hex_buf[['h3_index', 'geometry']],
        how='right', predicate='intersects'
    )
    hr_counts = hr_join.groupby('h3_index').size().reset_index(name='n_highrise')
else:
    hr_counts = pd.DataFrame({'h3_index': hex_gdf['h3_index'], 'n_highrise': 0})

# Count all buildings per cell (reuse for noise proxy in Cell 5)
all_join = gpd.sjoin(
    all_bldg[['geometry', '_floors']],
    hex_buf[['h3_index', 'geometry']],
    how='right', predicate='intersects'
)
all_counts   = all_join.groupby('h3_index').size().reset_index(name='n_buildings_total')
low_mask     = all_join['_floors'].fillna(999) < 5
lowrise_cnts = all_join[low_mask].groupby('h3_index').size().reset_index(name='n_lowrise')

hex_gdf = (hex_gdf
    .merge(hr_counts,   on='h3_index', how='left')
    .merge(all_counts,  on='h3_index', how='left')
    .merge(lowrise_cnts, on='h3_index', how='left'))
hex_gdf['n_highrise']       = hex_gdf['n_highrise'].fillna(0).astype(int)
hex_gdf['n_buildings_total']= hex_gdf['n_buildings_total'].fillna(0).astype(int)
hex_gdf['n_lowrise']        = hex_gdf['n_lowrise'].fillna(0).astype(int)

# Normalize high-rise count with p95 (not max)
p95_hr = hex_gdf['n_highrise'].quantile(0.95)
if p95_hr > 0:
    hex_gdf['score_obstacle'] = 1 - (hex_gdf['n_highrise'] / p95_hr).clip(0, 1)
else:
    hex_gdf['score_obstacle'] = 1.0

print(f'\n  n_highrise p95={p95_hr:.1f}')
print(f'  score_obstacle: mean={hex_gdf["score_obstacle"].mean():.3f}  '
      f'min={hex_gdf["score_obstacle"].min():.3f}  max={hex_gdf["score_obstacle"].max():.3f}')


  Layer used: buildings_5179
  Buildings loaded: 74,553  CRS=EPSG:5179
  Columns: ['순번', '건물관리번호', 'PNU', '법정동코드', '지번주소', '번지', '주부속구분코드', '대지구분코드', '건물용도코드', '건물용도명', '건물번호코드', '주구조', '건축면적_m2', '사용승인일', '연면적_m2', '대지면적_m2', '높이_m', '건폐율_pct', '용적률_pct', '도로명주소코드', '위반건물여부', '건물ID', '데이터기준일', '시군구코드', '건물명칭', '동명', '지상층수', '지하층수', '최종변경일', 'GU_NM', 'SHP_ADM_CD', 'ADM_NM', 'CSV_ADMI_CD', 'geometry']
  High-rise criterion: 지상층수 >= 15
  High-rise buildings: 1,589 / 74,553



  n_highrise p95=32.0
  score_obstacle: mean=0.843  min=0.000  max=0.969


In [5]:
# ═══════════════════════════════════════════════════════
#  소음 민감도 프록시 (Noise Sensitivity Proxy) — score_noise_proxy
#  ※ 공식 소음지도가 아닌 건물 밀도 기반 소프트 프록시입니다.
# ═══════════════════════════════════════════════════════

# Dense low-rise (residential) areas = higher complaint risk = lower score
# Building data already joined in Cell 4 (n_buildings_total, n_lowrise)

p95_total = hex_gdf['n_buildings_total'].quantile(0.95)
if p95_total > 0:
    density_norm = (hex_gdf['n_buildings_total'] / p95_total).clip(0, 1)
else:
    density_norm = pd.Series(0.0, index=hex_gdf.index)

lowrise_ratio = np.where(
    hex_gdf['n_buildings_total'] > 0,
    hex_gdf['n_lowrise'] / hex_gdf['n_buildings_total'],
    0.0
)

# Noise penalty: high density × high low-rise ratio = residential
noise_raw = density_norm.values * lowrise_ratio

# Min-Max normalize noise_raw
n_min, n_max = noise_raw.min(), noise_raw.max()
if n_max > n_min:
    noise_norm = (noise_raw - n_min) / (n_max - n_min)
else:
    noise_norm = np.zeros_like(noise_raw)

hex_gdf['noise_density_norm'] = noise_norm
hex_gdf['score_noise_proxy']  = 1.0 - noise_norm

print('[Noise Proxy] 건물 밀도 × 저층 비율 기반')
print(f'  p95 total buildings per cell: {p95_total:.1f}')
print(f'  score_noise_proxy: mean={hex_gdf["score_noise_proxy"].mean():.3f}  '
      f'min={hex_gdf["score_noise_proxy"].min():.3f}  '
      f'max={hex_gdf["score_noise_proxy"].max():.3f}')


[Noise Proxy] 건물 밀도 × 저층 비율 기반
  p95 total buildings per cell: 1100.7
  score_noise_proxy: mean=0.851  min=0.000  max=1.000


In [6]:
# ═══════════════════════════════════════════════════════
#  지형 경사도 (Terrain Slope) — score_terrain
# ═══════════════════════════════════════════════════════

slope_tif  = PROC / 'seongnam_slope_b010.tif'
slope_gpkg = PROC / 'slope_zones_b010.gpkg'
TERRAIN_OK = False

# ── Method A: rasterstats zonal_stats ─────────────────
if slope_tif.exists():
    try:
        from rasterstats import zonal_stats
        # Use hex geometry in native raster CRS
        # Read raster CRS first
        import rasterio
        with rasterio.open(slope_tif) as src:
            raster_crs = src.crs
            nodata_val = src.nodata
            print(f'  Slope raster CRS: {raster_crs}')
            print(f'  Nodata value    : {nodata_val}')

        hex_raster_crs = hex_gdf.to_crs(raster_crs) if raster_crs else hex_gdf
        stats = zonal_stats(
            hex_raster_crs.__geo_interface__,
            str(slope_tif),
            stats=['mean'],
            nodata=nodata_val if nodata_val is not None else -9999,
            all_touched=True
        )
        hex_gdf['avg_slope'] = [s['mean'] if s['mean'] is not None else np.nan
                                 for s in stats]
        n_null_slope = hex_gdf['avg_slope'].isna().sum()
        print(f'  zonal_stats OK: avg_slope computed  ({n_null_slope} null cells)')

        # Score: flat=1.0, moderate=0.6, steep=0.2
        # Thresholds from seongnam_slope_b010 convention:
        #   Green  0-8 degrees  → score 1.0
        #   Yellow 8-30 degrees → score 0.6
        #   Red    >30 degrees  → score 0.2
        def slope_to_score(deg):
            if pd.isna(deg): return 0.6   # fallback to moderate if missing
            if deg < 8:  return 1.0
            if deg < 30: return 0.6
            return 0.2

        hex_gdf['score_terrain'] = hex_gdf['avg_slope'].apply(slope_to_score)
        hex_gdf['terrain_method'] = 'rasterstats_zonal_mean'
        TERRAIN_OK = True

    except Exception as e:
        print(f'  rasterstats failed: {e}')

# ── Method B: slope_zones_b010.gpkg intersection ──────
if not TERRAIN_OK and slope_gpkg.exists():
    try:
        try:
            from pyogrio import list_layers as _ll
            slayer = _ll(str(slope_gpkg))[0][0]
        except Exception:
            slayer = None
        slope_zones = gpd.read_file(slope_gpkg) if slayer is None else gpd.read_file(slope_gpkg, layer=slayer)
        slope_zones = slope_zones.to_crs('EPSG:5179')
        print(f'  slope_zones layer: {slayer}  cols={list(slope_zones.columns)}')

        # Identify zone class column
        zone_col = None
        for c in ['zone_label', 'slope_class', 'class', 'color', 'label', 'LABEL']:
            if c in slope_zones.columns:
                zone_col = c
                break
        if zone_col is None:
            zone_col = slope_zones.columns[0]
            print(f'  WARNING: guessing zone column = {zone_col}')

        # Score mapping — try to match common labels
        def zone_to_score(label):
            l = str(label).lower().strip()
            if any(x in l for x in ['green', '녹', 'low', '0', 'gentle']): return 1.0
            if any(x in l for x in ['yellow', '황', 'mod', '1', 'medium']): return 0.6
            if any(x in l for x in ['red', '적', 'steep', '2', 'high']):    return 0.2
            return 0.6  # default moderate

        slope_zones['_terrain_score'] = slope_zones[zone_col].apply(zone_to_score)

        # Largest-overlap intersection per H3 cell
        intr = gpd.overlay(hex_5179.reset_index(), slope_zones[['geometry', '_terrain_score']],
                           how='intersection')
        intr['_area'] = intr.geometry.area
        dominant = (intr.sort_values('_area', ascending=False)
                       .groupby('h3_index').first()
                       .reset_index()[['h3_index', '_terrain_score']])
        hex_gdf = hex_gdf.merge(dominant, on='h3_index', how='left')
        hex_gdf['score_terrain'] = hex_gdf['_terrain_score'].fillna(0.6)
        hex_gdf.drop(columns=['_terrain_score'], inplace=True, errors='ignore')
        hex_gdf['terrain_method'] = 'slope_zones_intersection'
        TERRAIN_OK = True
        print(f'  slope_zones intersection OK')
    except Exception as e:
        print(f'  slope_zones intersection failed: {e}')

# ── Fallback: warn clearly ─────────────────────────────
if not TERRAIN_OK:
    print('  WARNING: Both terrain methods failed.')
    print('  score_terrain set to 0.6 (moderate) — treat with caution!')
    hex_gdf['score_terrain']  = 0.6
    hex_gdf['avg_slope']      = np.nan
    hex_gdf['terrain_method'] = 'fallback_moderate'

print(f'\n  score_terrain: mean={hex_gdf["score_terrain"].mean():.3f}  '
      f'min={hex_gdf["score_terrain"].min():.3f}  '
      f'max={hex_gdf["score_terrain"].max():.3f}')
print(f'  terrain_method: {hex_gdf["terrain_method"].value_counts().to_dict()}')


  Slope raster CRS: EPSG:5179
  Nodata value    : nan


  zonal_stats OK: avg_slope computed  (0 null cells)

  score_terrain: mean=0.726  min=0.200  max=1.000
  terrain_method: {'rasterstats_zonal_mean': 1947}


In [7]:
# ═══════════════════════════════════════════════════════
#  로봇 접근성 (Robot Access) — score_robot
# ═══════════════════════════════════════════════════════

robot_path = PROC / 'robot_access_grid.gpkg'
if not robot_path.exists():
    print('  WARNING: robot_access_grid.gpkg not found — score_robot = 0.0')
    hex_gdf['score_robot'] = 0.0
    hex_gdf['Ra']          = np.nan
else:
    robot_gdf = gpd.read_file(robot_path)
    print(f'  Robot grid: {len(robot_gdf)} cells  cols={list(robot_gdf.columns)}')

    # Alias h3_id → h3_index
    if 'h3_index' not in robot_gdf.columns and 'h3_id' in robot_gdf.columns:
        robot_gdf['h3_index'] = robot_gdf['h3_id']

    robot_cols = ['h3_index', 'Ra']
    for opt in ['Ra_grade', 'total_walk_length', 'robot_walk_length',
                'n_crossings', 'avg_robot_weight']:
        if opt in robot_gdf.columns:
            robot_cols.append(opt)

    # ── Resolution-aware merge ─────────────────────────────────
    # robot_access_grid.gpkg may be at a different H3 resolution (e.g. res 8)
    # than the current demand grid (e.g. res 9). Detect and handle via
    # h3 parent lookup if direct merge finds no matches.
    common = set(hex_gdf['h3_index']) & set(robot_gdf['h3_index'])
    if len(common) > 0:
        print(f'  Direct h3_index match: {len(common)} cells — using direct merge')
        hex_gdf = hex_gdf.merge(
            pd.DataFrame(robot_gdf[robot_cols]),
            on='h3_index', how='left'
        )
    else:
        # Parent lookup: map each res-9 cell to its res-8 parent
        print(f'  No direct h3_index match — using parent cell lookup (res 9 → res 8)')
        try:
            hex_gdf['_parent_h3'] = hex_gdf['h3_index'].apply(
                lambda h: h3.cell_to_parent(h, 8))
        except AttributeError:
            hex_gdf['_parent_h3'] = hex_gdf['h3_index'].apply(
                lambda h: h3.h3_to_parent(h, 8))

        robot_for_merge = pd.DataFrame(robot_gdf[robot_cols]).rename(
            columns={'h3_index': '_parent_h3'})
        hex_gdf = hex_gdf.merge(robot_for_merge, on='_parent_h3', how='left')
        hex_gdf.drop(columns=['_parent_h3'], inplace=True)

        n_matched = hex_gdf['Ra'].notna().sum()
        print(f'  Parent lookup matched: {n_matched}/{len(hex_gdf)} cells')

    hex_gdf['score_robot'] = hex_gdf['Ra'].fillna(0.0)
    n_null = hex_gdf['Ra'].isna().sum()
    print(f'  Ra null cells: {n_null}')
    print(f'  score_robot: mean={hex_gdf["score_robot"].mean():.3f}  '
          f'min={hex_gdf["score_robot"].min():.3f}  '
          f'max={hex_gdf["score_robot"].max():.3f}')


  Robot grid: 283 cells  cols=['h3_index', 'lat', 'lon', 'ADM_NM', 'GU_NM', 'CSV_ADMI_CD', 'Ds', 'demand_grade', 'Ra', 'Ra_grade', 'Ra_percentile', 'total_walk_length', 'robot_walk_length', 'n_crossings', 'avg_robot_weight', 'ratio_footway', 'has_crossing', 'ratio_footway_norm', 'walk_density_norm', 'geometry']
  No direct h3_index match — using parent cell lookup (res 9 → res 8)
  Parent lookup matched: 1906/1947 cells
  Ra null cells: 41
  score_robot: mean=0.332  min=0.000  max=0.877


In [8]:
# ═══════════════════════════════════════════════════════
#  기상 위험도 (Weather Risk) — score_weather
# ═══════════════════════════════════════════════════════

weather_path = PROC / 'weather_risk_grid.gpkg'
if not weather_path.exists():
    print('  WARNING: weather_risk_grid.gpkg not found — score_weather = 0.944 (1-year obs default)')
    hex_gdf['score_weather'] = 0.944
    hex_gdf['weather_risk']  = 0.056
else:
    weather_gdf = gpd.read_file(weather_path)
    print(f'  Weather grid: {len(weather_gdf)} cells  cols={list(weather_gdf.columns)}')

    if 'h3_index' not in weather_gdf.columns and 'h3_id' in weather_gdf.columns:
        weather_gdf['h3_index'] = weather_gdf['h3_id']

    weather_cols = ['h3_index', 'score_weather']
    for opt in ['weather_risk', 'P_wind_fail', 'P_rain_fail',
                'P_visibility_fail', 'weather_source', 'weather_period_start']:
        if opt in weather_gdf.columns:
            weather_cols.append(opt)

    # ── Resolution-aware merge ─────────────────────────────────
    # Weather data is station-level (uniform across all cells), so if
    # h3 resolution changed, we can safely broadcast the value.
    common = set(hex_gdf['h3_index']) & set(weather_gdf['h3_index'])
    if len(common) > 0:
        print(f'  Direct h3_index match: {len(common)} cells — using direct merge')
        hex_gdf = hex_gdf.merge(
            pd.DataFrame(weather_gdf[weather_cols]),
            on='h3_index', how='left'
        )
    else:
        # Weather is station-level uniform — broadcast directly
        sw_val = float(weather_gdf['score_weather'].median())
        print(f'  No direct h3_index match — weather is station-uniform')
        print(f'  Broadcasting score_weather={sw_val:.4f} to all {len(hex_gdf)} cells')
        hex_gdf['score_weather'] = sw_val

        # Propagate metadata columns (all uniform for station-level data)
        for opt in ['weather_risk', 'P_wind_fail', 'P_rain_fail',
                    'P_visibility_fail']:
            if opt in weather_gdf.columns:
                hex_gdf[opt] = float(weather_gdf[opt].median())
        for opt in ['weather_source', 'weather_period_start']:
            if opt in weather_gdf.columns:
                hex_gdf[opt] = str(weather_gdf[opt].iloc[0])

    # Fill null with median (safety net)
    sw_median = hex_gdf['score_weather'].median()
    hex_gdf['score_weather'] = hex_gdf['score_weather'].fillna(sw_median)
    print(f'  score_weather: mean={hex_gdf["score_weather"].mean():.4f}  '
          f'(station-level uniform across all cells)')
    if 'weather_source' in hex_gdf.columns:
        print(f'  weather_source: {hex_gdf["weather_source"].dropna().unique()}')


  Weather grid: 283 cells  cols=['h3_index', 'lat', 'lon', 'ADM_NM', 'GU_NM', 'CSV_ADMI_CD', 'Ds', 'demand_grade', 'P_wind_fail', 'P_rain_fail', 'P_visibility_fail', 'P_drone_weather_available', 'weather_risk', 'score_weather', 'weather_station', 'weather_source', 'weather_period_start', 'weather_period_end', 'weather_note', 'geometry']
  No direct h3_index match — weather is station-uniform
  Broadcasting score_weather=0.9441 to all 1947 cells
  score_weather: mean=0.9441  (station-level uniform across all cells)
  weather_source: ['kma_hourly_observed_1year']


In [9]:
# ═══════════════════════════════════════════════════════
#  통합 제약 점수 (Composite Constraint Score)
#
#  설계 원칙: constraint_score = 물리적 환경 품질만 반영.
#  공역제한(관제권)은 규제 레이어 → hard_exclusion_flag 로 별도 추적.
#  원가중치 (airspace 포함) 0.20/0.20/0.15/0.20/0.15/0.10 에서
#  airspace 0.20 제거 후 나머지 0.80 으로 나눠 재정규화.
# ═══════════════════════════════════════════════════════

W_OBSTACLE = 0.25       # 0.20 / 0.80
W_NOISE    = 0.1875     # 0.15 / 0.80
W_TERRAIN  = 0.25       # 0.20 / 0.80
W_ROBOT    = 0.1875     # 0.15 / 0.80
W_WEATHER  = 0.125      # 0.10 / 0.80
assert abs(W_OBSTACLE + W_NOISE + W_TERRAIN + W_ROBOT + W_WEATHER - 1.0) < 1e-9, \
    "Weights do not sum to 1.0"

hex_gdf['constraint_score'] = (
    W_OBSTACLE * hex_gdf['score_obstacle']
  + W_NOISE    * hex_gdf['score_noise_proxy']
  + W_TERRAIN  * hex_gdf['score_terrain']
  + W_ROBOT    * hex_gdf['score_robot']
  + W_WEATHER  * hex_gdf['score_weather']
)

# Clamp to [0, 1] (guard against float precision)
hex_gdf['constraint_score'] = hex_gdf['constraint_score'].clip(0.0, 1.0)

print(f'[Composite Score] — 물리적 환경 품질 (공역제한 제외)')
print(f'  constraint_score: mean={hex_gdf["constraint_score"].mean():.3f}  '
      f'min={hex_gdf["constraint_score"].min():.3f}  '
      f'max={hex_gdf["constraint_score"].max():.3f}')
print()
print(f'  가중치 (재정규화, 합=1.0):')
print(f'    score_obstacle    × {W_OBSTACLE}')
print(f'    score_noise_proxy × {W_NOISE}')
print(f'    score_terrain     × {W_TERRAIN}')
print(f'    score_robot       × {W_ROBOT}')
print(f'    score_weather     × {W_WEATHER}')
print()
print(f'  score_airspace → 규제 레이어 (hard_exclusion_flag 로 별도 추적, 점수에 미포함)')

[Composite Score] — 물리적 환경 품질 (공역제한 제외)
  constraint_score: mean=0.732  min=0.444  max=0.940

  가중치 (재정규화, 합=1.0):
    score_obstacle    × 0.25
    score_noise_proxy × 0.1875
    score_terrain     × 0.25
    score_robot       × 0.1875
    score_weather     × 0.125

  score_airspace → 규제 레이어 (hard_exclusion_flag 로 별도 추적, 점수에 미포함)


In [10]:
# ═══════════════════════════════════════════════════════
#  등급 부여 (Percentile Grades A/B/C/D)
# ═══════════════════════════════════════════════════════

cs = hex_gdf['constraint_score']
q75 = cs.quantile(0.75)
q50 = cs.quantile(0.50)
q25 = cs.quantile(0.25)

def _grade(s):
    if s >= q75: return 'A'
    if s >= q50: return 'B'
    if s >= q25: return 'C'
    return 'D'

hex_gdf['constraint_grade'] = hex_gdf['constraint_score'].apply(_grade)

# Hard exclusion flag — only meaningful when official airspace data was used.
# 'official_vworld_wfs_관제권' : NB07d WFS 관제권 (현재 공식 소스)
# 'official_local'             : 기타 로컬 공식 파일
# 'official_vworld'            : 구 VWorld Data API 방식 (레거시)
OFFICIAL_STATUSES = ('official_vworld_wfs_관제권', 'official_local', 'official_vworld')
has_official = airspace_data_status in OFFICIAL_STATUSES
if has_official:
    hex_gdf['hard_exclusion_flag'] = hex_gdf['score_airspace'] == 0.0
else:
    hex_gdf['hard_exclusion_flag'] = False  # 공식 데이터 없이는 판단 불가

# constraint_note: 각 레이어의 데이터 출처 기록
notes = []
notes.append(f'airspace:{airspace_data_status}')
notes.append('obstacle:official_osm_buildings')
notes.append('noise:proxy_building_density')
notes.append(f'terrain:{hex_gdf["terrain_method"].iloc[0]}')
notes.append('robot:NB07b_ra_index')
notes.append('weather:NB07c_kma_1year')
hex_gdf['constraint_note'] = ' | '.join(notes)

print('[Grades]')
print(hex_gdf['constraint_grade'].value_counts().sort_index().to_string())
print(f'\n  q25={q25:.3f}  q50={q50:.3f}  q75={q75:.3f}')
print(f'  has_official_airspace: {has_official}  (status={airspace_data_status})')
print(f'  hard_exclusion_flag  : {hex_gdf["hard_exclusion_flag"].sum()} 셀')

[Grades]
constraint_grade
A    487
B    487
C    486
D    487

  q25=0.696  q50=0.740  q75=0.764
  has_official_airspace: True  (status=official_vworld_wfs_관제권)
  hard_exclusion_flag  : 1688 셀


In [11]:
# ═══════════════════════════════════════════════════════
#  저장
# ═══════════════════════════════════════════════════════

# Final column order
SCORE_COLS = [
    'score_airspace',                          # 규제 메타데이터 (composite 에 미포함)
    'score_obstacle', 'score_noise_proxy',
    'score_terrain', 'score_robot', 'score_weather',
    'constraint_score', 'constraint_grade',
]
META_COLS = [
    'n_highrise', 'n_buildings_total', 'n_lowrise',
    'Ra', 'Ra_grade', 'n_crossings',
    'weather_risk', 'weather_source',
    'airspace_data_status', 'airspace_proxy_flag', 'terrain_method',
    'hard_exclusion_flag', 'constraint_note',
]

BASE_COLS = ['h3_index', 'lat', 'lon', 'CSV_ADMI_CD', 'ADM_NM', 'GU_NM', 'Ds', 'demand_grade']
FINAL_COLS = [c for c in BASE_COLS + SCORE_COLS + META_COLS if c in hex_gdf.columns]

out_gdf = gpd.GeoDataFrame(hex_gdf[FINAL_COLS + ['geometry']], crs=hex_gdf.crs)

# Drop OGR case-conflict lowercase aliases
for col in ['gu_nm', 'adm_nm', 'dong_nm', 'h3_id']:
    if col in out_gdf.columns:
        out_gdf.drop(columns=[col], inplace=True)
# Cast object columns to string for GPKG
for col in out_gdf.select_dtypes(include=['object', 'bool']).columns:
    if col != 'geometry':
        out_gdf[col] = out_gdf[col].astype(str)

# Primary outputs
gpkg_path  = PROC / 'constraint_grid.gpkg'
csv_path   = PROC / 'constraint_grid.csv'
# Compatibility copies
compat_gpkg = PROC / 'constraint_layers.gpkg'
compat_csv  = PROC / 'constraint_layers.csv'

out_gdf.to_file(gpkg_path, driver='GPKG')
pd.DataFrame(out_gdf.drop(columns=['geometry'])).to_csv(csv_path, index=False, encoding='utf-8-sig')
out_gdf.to_file(compat_gpkg, driver='GPKG')
pd.DataFrame(out_gdf.drop(columns=['geometry'])).to_csv(compat_csv, index=False, encoding='utf-8-sig')

# Layer summary
score_cols_present = [
    c for c in SCORE_COLS
    if c in hex_gdf.columns and pd.api.types.is_numeric_dtype(hex_gdf[c])
]
summary_rows = []
for sc in score_cols_present:
    s = hex_gdf[sc]
    summary_rows.append({
        'score_column': sc,
        'mean': round(float(s.mean()), 4),
        'std':  round(float(s.std()),  4),
        'min':  round(float(s.min()),  4),
        'max':  round(float(s.max()),  4),
        'n_null': int(s.isna().sum()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(PROC / 'constraint_layer_summary.csv', index=False, encoding='utf-8-sig')

for p in [gpkg_path, csv_path, compat_gpkg, compat_csv,
          PROC / 'constraint_layer_summary.csv']:
    sz = p.stat().st_size // 1024
    print(f'  Saved: {p.name}  ({sz} KB)')

  Saved: constraint_grid.gpkg  (1500 KB)
  Saved: constraint_grid.csv  (955 KB)
  Saved: constraint_layers.gpkg  (1500 KB)
  Saved: constraint_layers.csv  (955 KB)
  Saved: constraint_layer_summary.csv  (0 KB)


In [12]:
# ═══════════════════════════════════════════════════════
#  지도 시각화 (Folium choropleth)
# ═══════════════════════════════════════════════════════
try:
    import folium
    from branca.colormap import LinearColormap

    cen_lat = hex_gdf['lat'].mean() if 'lat' in hex_gdf.columns else 37.42
    cen_lon = hex_gdf['lon'].mean() if 'lon' in hex_gdf.columns else 37.42

    fmap = folium.Map(location=[cen_lat, 127.13], zoom_start=12, tiles='CartoDB positron')

    cmap = LinearColormap(['#d73027', '#fee090', '#1a9641'],
                           vmin=hex_gdf['constraint_score'].min(),
                           vmax=hex_gdf['constraint_score'].max(),
                           caption='제약 종합 점수 (constraint_score)')
    cmap.add_to(fmap)

    hex_wgs84 = out_gdf.to_crs('EPSG:4326')
    for _, row in hex_wgs84.iterrows():
        score = row.get('constraint_score', 0)
        grade = row.get('constraint_grade', '?')
        adm   = row.get('ADM_NM', '')
        tip   = (f'{adm}<br>'
                 f'constraint: {score:.3f} ({grade})<br>'
                 f'obstacle: {row.get("score_obstacle", float("nan")):.3f}  '
                 f'terrain: {row.get("score_terrain", float("nan")):.3f}  '
                 f'robot: {row.get("score_robot", float("nan")):.3f}')
        folium.GeoJson(
            mapping(row.geometry),
            style_function=lambda feat, s=score: {
                'fillColor': cmap(s),
                'color': 'white',
                'weight': 0.3,
                'fillOpacity': 0.75,
            },
            tooltip=folium.Tooltip(tip),
        ).add_to(fmap)

    map_path = PROC / 'constraint_grid_map.html'
    fmap.save(str(map_path))
    print(f'  Map saved: {map_path.name}  ({map_path.stat().st_size // 1024} KB)')
except Exception as e:
    print(f'  Map skipped: {e}')


  Map saved: constraint_grid_map.html  (3351 KB)


In [13]:
# ═══════════════════════════════════════════════════════
#  차트 시각화
# ═══════════════════════════════════════════════════════
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.font_manager as fm

    plt.rcParams['axes.unicode_minus'] = False
    for fname in ['Malgun Gothic', 'NanumGothic']:
        try:
            fm.findfont(fname, fallback_to_default=False)
            plt.rcParams['font.family'] = fname
            break
        except Exception:
            pass

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('NB09 — 제약 레이어 점수 분포', fontsize=13, fontweight='bold')

    score_pairs = [
        ('score_airspace',    '공역제한',    axes[0, 0]),
        ('score_obstacle',    '장애물',      axes[0, 1]),
        ('score_noise_proxy', '소음 프록시', axes[0, 2]),
        ('score_terrain',     '지형 경사',   axes[1, 0]),
        ('score_robot',       '로봇 접근성', axes[1, 1]),
        ('constraint_score',  '종합 점수',   axes[1, 2]),
    ]

    COLORS = ['#7B68EE', '#E57373', '#FFB74D', '#4CAF50', '#29B6F6', '#AB47BC']
    for (col, label, ax), color in zip(score_pairs, COLORS):
        if col not in hex_gdf.columns:
            ax.set_title(f'{label} (없음)')
            continue
        vals = hex_gdf[col].dropna()
        ax.hist(vals, bins=25, color=color, alpha=0.8, edgecolor='white')
        ax.axvline(vals.mean(), color='black', ls='--', lw=1.2,
                   label=f'평균 {vals.mean():.3f}')
        ax.set_title(label)
        ax.set_xlabel('점수')
        ax.set_ylabel('셀 수')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    chart_path = PROC / 'constraint_chart.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    print(f'  Chart saved: {chart_path.name}')
except Exception as e:
    print(f'  Chart skipped: {e}')


  Chart saved: constraint_chart.png


In [14]:
# ═══════════════════════════════════════════════════════
#  최종 검증 (Validation)
# ═══════════════════════════════════════════════════════

print('=' * 65)
print('NB09 최종 검증')
print('=' * 65)

# Row count
import geopandas as _gpd
demand_rows = len(_gpd.read_file(PROC / 'demand_grid.gpkg'))
print(f'\n[Row count]')
print(f'  demand_grid.gpkg : {demand_rows}')
print(f'  constraint_grid  : {len(out_gdf)}')
print(f'  Match: {demand_rows == len(out_gdf)}')

# Null counts for key columns
KEY_COLS = [
    'h3_index', 'ADM_NM', 'GU_NM', 'CSV_ADMI_CD', 'Ds',
    'score_airspace', 'score_obstacle', 'score_noise_proxy',
    'score_terrain', 'score_robot', 'score_weather',
    'constraint_score',
]
print('\n[Null counts]')
for c in KEY_COLS:
    if c in out_gdf.columns:
        n = out_gdf[c].isna().sum() if out_gdf[c].dtype != object else (out_gdf[c] == 'nan').sum()
        flag = '  <-- PROBLEM' if n > 0 else ''
        print(f'  {c:<30} {n}{flag}')
    else:
        print(f'  {c:<30} COLUMN MISSING')

# Score ranges
SCORE_COLS_CHECK = [
    'score_airspace', 'score_obstacle', 'score_noise_proxy',
    'score_terrain', 'score_robot', 'score_weather', 'constraint_score',
]
print('\n[Score ranges — all must be [0, 1]]')
all_in_range = True
for sc in SCORE_COLS_CHECK:
    if sc not in hex_gdf.columns:
        print(f'  {sc:<30} MISSING')
        continue
    s = hex_gdf[sc]
    lo, hi = float(s.min()), float(s.max())
    ok = (lo >= -1e-9) and (hi <= 1 + 1e-9)
    if not ok:
        all_in_range = False
    flag = '' if ok else '  <-- OUT OF RANGE'
    print(f'  {sc:<30} [{lo:.4f}, {hi:.4f}]  mean={s.mean():.4f}{flag}')
print(f'  All scores in [0,1]: {all_in_range}')

# Top 10 and bottom 10
print('\n[Top 10 by constraint_score]')
top10 = hex_gdf.nlargest(10, 'constraint_score')[
    ['ADM_NM', 'GU_NM', 'constraint_score', 'constraint_grade',
     'score_airspace', 'score_obstacle', 'score_terrain', 'score_robot']
].round(3)
print(top10.to_string(index=False))

print('\n[Bottom 10 by constraint_score]')
bot10 = hex_gdf.nsmallest(10, 'constraint_score')[
    ['ADM_NM', 'GU_NM', 'constraint_score', 'constraint_grade',
     'score_airspace', 'score_obstacle', 'score_terrain', 'score_robot']
].round(3)
print(bot10.to_string(index=False))

# Airspace status summary
print('\n[공역 데이터 상태]')
print(f'  airspace_data_status : {airspace_data_status}')
print(f'  airspace_source      : {airspace_source}')
print(f'  AIRSPACE_PROXY_FLAG  : {AIRSPACE_PROXY_FLAG}')

print('\n✅ NB09 검증 완료')
print('  constraint_grid.gpkg  / constraint_grid.csv')
print('  constraint_layers.gpkg / constraint_layers.csv  (호환 사본)')
print('  constraint_layer_summary.csv')


NB09 최종 검증

[Row count]
  demand_grid.gpkg : 1947
  constraint_grid  : 1947
  Match: True

[Null counts]
  h3_index                       0
  ADM_NM                         0
  GU_NM                          0
  CSV_ADMI_CD                    0
  Ds                             0
  score_airspace                 0
  score_obstacle                 0
  score_noise_proxy              0
  score_terrain                  0
  score_robot                    0
  score_weather                  0
  constraint_score               0

[Score ranges — all must be [0, 1]]
  score_airspace                 [0.0000, 1.0000]  mean=0.1330
  score_obstacle                 [0.0000, 0.9688]  mean=0.8429
  score_noise_proxy              [0.0000, 1.0000]  mean=0.8510
  score_terrain                  [0.2000, 1.0000]  mean=0.7255
  score_robot                    [0.0000, 0.8773]  mean=0.3317
  score_weather                  [0.9441, 0.9441]  mean=0.9441
  constraint_score               [0.4442, 0.9397]  mean=0.73